# Why JSON Nodes?

XQuery 3.1 gave us maps and arrays — first-class JSON-compatible data structures. You can parse JSON, build maps, and serialize back to JSON:

In [ ]:
let $data := map {
    "name": "Ada Lovelace",
    "born": 1815,
    "fields": array { "mathematics", "computing" }
}
return $data?name

This works well for simple lookups. But what if you want to *navigate* a JSON structure the way you navigate XML — with XPath axes, predicates, and multi-step paths?

In XQuery 3.1, you can't. Maps and arrays are opaque containers: you look things up by key or index, but you can't ask "give me all the descendants" or "what is the parent of this value?"

## The XQuery 4.0 Answer

XQuery 4.0 introduces **JSON nodes** (JNodes) — a new node type that brings XPath's navigation model to JSON data. A JNode tree mirrors the structure of a map or array, but each value becomes a node with a parent, children, siblings, and a position in document order.

Here is a map and its JNode equivalent side by side. The map on the left supports lookup (`?key`). The JNode tree on the right supports both lookup and full axis navigation:

In [ ]:
xquery version "4.0";

(: A plain map — supports lookup only :)
let $map := map { "a": 1, "b": map { "x": 10 } }

(: The same structure as a JNode tree — supports XPath navigation :)
let $tree := fn:jtree($map)

return map {
    "map lookup": $map?b?x,
    "jnode child count": count($tree/child::*),
    "jnode descendant count": count($tree/descendant::*)
}

The [`fn:jtree()`]({docs}/functions/fn/jtree) function converts a map or array into a JNode tree. Once you have a tree, all XPath axes work — `child`, `parent`, `descendant`, `ancestor`, `following-sibling`, and more.

## What Kind of Nodes?

JNodes come in several kinds, mirroring the JSON value types:

| JSON value | JNode kind | Kind test |
|------------|-----------|-----------|
| `{ ... }` | object node | `object-node()` |
| `[ ... ]` | array node | `array-node()` |
| `"text"` | string node | `string-node()` |
| `42`, `3.14` | number node | `number-node()` |
| `true`/`false` | boolean node | `boolean-node()` |
| `null` | null node | `null-node()` |

Try it — this query shows the kind of each child node in a JNode tree:

In [ ]:
xquery version "4.0";

let $tree := fn:jtree(map {
    "name": "eXist-db",
    "version": 7,
    "open-source": true(),
    "modules": array { "lucene", "ft", "websocket" }
})
for $child in $tree/child::*
return fn:jkey($child) || " -> " || string-join(fn:jvalue($child), ", ")

Each child has a **key** (from the map entry) and a **value** (the actual data). The [`fn:jkey()`]({docs}/functions/fn/jkey) and [`fn:jvalue()`]({docs}/functions/fn/jvalue) functions extract these.